In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

In [2]:
from app.app import load_resources, load_semantic_resources

documents, bm25 = load_resources()

2026-04-22 09:56:23.935 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:56:23.936 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:56:23.999 
  command:

    streamlit run /Users/randalllee/miniforge3/envs/dsci-575-project/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-22 09:56:24.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:56:24.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:56:24.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:56:24.001 Thread 'MainThread': missing ScriptRunContext! 

In [3]:
unique_asins = {doc.metadata["asin"] for doc in documents}

print(len(unique_asins))

10000


In [4]:
documents, vectorstore = load_semantic_resources()

2026-04-22 09:56:24.268 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
unique_asins = {doc.metadata["asin"] for doc in documents}

print(len(unique_asins))

10000


### Testing current Meta-Llama-3-8B-Instruct model vs alternative Qwen/Qwen3.5-9B model

In [6]:
from dotenv import load_dotenv
import os

In [39]:
from src.semantic import create_faiss_index
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

vectorstore = create_faiss_index(documents, embedding_model, sample_size=10000, reload_index=False)


retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('asin')}\n"
        f"Product Title: {doc.metadata.get('product_title')}\n"
        f"Product Rating: {doc.metadata.get('product_rating')}\n"
        f"Product Review: {doc.metadata.get('product_review')}\n"
        for doc in docs
    )

format_context = RunnableLambda(build_context)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

def get_query_response(query, model_name):
    llm_endpoint = HuggingFaceEndpoint(
        repo_id=model_name,
        task="text-generation", # Keep this as text-generation for the base
        max_new_tokens=100,
        huggingfacehub_api_token=token,
        temperature=0.0,
        provider="auto" #"novita"
        )

    llm = ChatHuggingFace(llm=llm_endpoint)

    prompt = ChatPromptTemplate.from_template(
        """
        You are a helpful Amazon shopping assistant.

        You must answer using ONLY the information in the context.

        - Recommend ONE product.
        - Do NOT use outside knowledge.
        - Do NOT include any extra text.
        - Return ONLY valid JSON.

        Context:
        {context}

        Question:
        {input}

        Return exactly in this format:

        {{
        "product_title": "",
        "product_asin": "",
        "product_rating": "",
        "product_review": "",
        "reason_for_recommendation": ""
        }}
        """
        )

    rag_chain = (
        {
            "context": retriever | format_context,
            "input": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain.invoke(query)

In [40]:
queries = [
    "moisturizing shampoo for thick curly hair",
    "best product for dry skin",
    "something gentle for sensitive skin",
    "ultra facial barrier-hydrating cleanser",
    "best sunscreen for scuba diving in tropical regions"
]

In [41]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

for q in queries:
    print("Query:", q)
    print(get_query_response(q, model_name))
    print("-" * 50)


Query: moisturizing shampoo for thick curly hair
{
"product_title": "Marc Anthony Instantly Thick Volumizing Conditioner 12.9oz (6 Pack)",
"product_asin": "B097545V9C",
"product_rating": "3.0",
"product_review": "i have curly hair that is going a little thin. This stuff was ok. The shampoo isn't very foamy and the conditioner needs more 'slip' My hair felt a teeny bit fuller when dry, but nothing significant. I
--------------------------------------------------
Query: best product for dry skin
{
"product_title": "CeraVe Moisturizing Cream and Healing Ointment (1.89 oz) Bundle - Choose a 12 oz Tub or A 19 oz Tub (19 oz Tub with Healing Ointment)",
"product_asin": "B098YVDR76",
"product_rating": "5.0",
"product_review": "Great product that my doctor recommended for very dry skin.",
"reason_for_recommendation": "Recommended by a doctor for dry skin
--------------------------------------------------
Query: something gentle for sensitive skin
{
"product_title": "Le Petit Marseillais Peony a

In [42]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

for q in queries:
    print("Query:", q)
    print(get_query_response(q, model_name))
    print("-" * 50)

Query: moisturizing shampoo for thick curly hair
{
"product_title": "Detangler Brush Natural Boar Bristle, Set of 2, for Men, Women or Kids with Thick or Curly Hair",
"product_asin": "B07G7B3F1F",
"product_rating": "5.0",
"product_review": "I have thick curly hair and this brush is great. I have broken many brushes. I use it with wet hair and it detangles like no other brush I have used.",

--------------------------------------------------
Query: best product for dry skin
{
"product_title": "Lumene Vitamin C+ Dry Skin Comfort Radiance Cocktail - 1 fl oz.",
"product_asin": "B004GHIEZQ",
"product_rating": "5.0",
"product_review": "Since moving to a very dry climate, my skin has been constantly dry. I'm always on the search for moisturizing products. I've tried expensive products, basic oils and everything in between. This is the very first product to keep
--------------------------------------------------
Query: something gentle for sensitive skin
{
"product_title": "Kastu Foot Peel Mas